In [1]:
import os
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots 
import h5py
import pandas as pd

In [2]:
pd.options.plotting.backend = "plotly"

In [3]:
%load_ext autoreload
%autoreload 2

In [11]:
filenames = [r"D:\Data\Foils\Testing_001.h5",
             r"D:\Data\Foils\Testing_002.h5"]

In [12]:
def show_info(name, h5obj):
    print("{}: {}".format(name, h5obj))
    for attrname, attrval in h5obj.attrs.items():
        print("\t{}: {}".format(attrname, attrval))
    

In [13]:
with h5py.File(filenames[0], 'r') as h5file:
    h5file.visititems(show_info)

Calibrated: <HDF5 group "/Calibrated" (8 members)>
Calibrated/CalibrationMatrix: <HDF5 dataset "CalibrationMatrix": shape (6, 6), type "<f8">
Calibrated/Encoder: <HDF5 dataset "Encoder": shape (4334,), type "<f8">
	CountsPerRev: 10000
Calibrated/xForce: <HDF5 dataset "xForce": shape (4334,), type "<f8">
Calibrated/xTorque: <HDF5 dataset "xTorque": shape (4334,), type "<f8">
Calibrated/yForce: <HDF5 dataset "yForce": shape (4334,), type "<f8">
Calibrated/yTorque: <HDF5 dataset "yTorque": shape (4334,), type "<f8">
Calibrated/zForce: <HDF5 dataset "zForce": shape (4334,), type "<f8">
Calibrated/zTorque: <HDF5 dataset "zTorque": shape (4334,), type "<f8">
NominalStimulus: <HDF5 group "/NominalStimulus" (4 members)>
	Amplitude: 25
	Cycles: 10
	Frequency: 3
	ScaleFactor: 1
	Type: Constant Frequency
	WaitPost: 0.0
	WaitPre: 1.0
NominalStimulus/Position: <HDF5 dataset "Position": shape (4334,), type "<f8">
	Units: deg
NominalStimulus/Velocity: <HDF5 dataset "Velocity": shape (4334,), type "<f

In [24]:
class FlapperData:
    def __init__(self, filename):
        with h5py.File(filename, 'r') as h5file:
            self.yforce = np.array(h5file['/Calibrated/yForce'])
            self.xtorque = np.array(h5file['/Calibrated/xTorque'])
            self.ztorque = np.array(h5file['/Calibrated/zTorque'])
            self.t = np.array(h5file['/NominalStimulus/t'])
            self.tnorm = np.array(h5file['/NominalStimulus/tnorm'])
            self.angle = np.array(h5file['/Calibrated/Encoder'])
            self.angle_cmd = np.array(h5file['/NominalStimulus/Position'])

            self.flowspeed_mm = h5file.attrs['FlowSpeed_mm']
            self.foillen_mm = h5file.attrs['Foillen_mm']
            self.ncycles = h5file['/NominalStimulus'].attrs['Cycles']
            self.freq = h5file['/NominalStimulus'].attrs['Frequency']

    def get_cycle_means(self):
        isearly = self.t < -0.5
        yfbaseline = np.mean(self.yforce[isearly])
        self.yforce = self.yforce - yfbaseline

        xtbaseline = np.mean(self.xtorque[isearly])
        self.xtorque = self.xtorque - xtbaseline

        meanyforce = []
        meanxtorque = []

        for c in range(0, self.ncycles):
            iscycle = (self.tnorm >= c) & (self.tnorm < c+1)

            meanyforce1 = np.mean(self.yforce[iscycle])
            meanyforce.append(meanyforce1)

            meanxtorque1 = np.mean(self.xtorque[iscycle])
            meanxtorque.append(meanxtorque1)

        self.tmean = np.arange(0, self.ncycles) / self.freq
        self.meanyforce = np.array(meanyforce)        
        self.meanxtorque = np.array(meanxtorque)        


In [25]:
data = []
for fn in filenames:
    fd1 = FlapperData(fn)
    fd1.get_cycle_means()
    data.append(fd1)

In [27]:
data[0].meanyforce

array([0.36314956, 0.42054859, 0.39589892, 0.40323444, 0.40123295,
       0.39916243, 0.39554867, 0.39526305, 0.39326094, 0.37351665])

In [38]:
meanyforce = np.array([np.mean(d.meanyforce[1:]) for d in data])
flowspeeds = np.array([d.flowspeed_mm for d in data])

In [ ]:
flowspeeds

[280, 150]

In [41]:
#fit line
m, b = np.polyfit(flowspeeds, meanyforce, 1)

xintercept = -b / m

# Generate points for the fitted line
xvals = np.append(flowspeeds, xintercept)
predforce = m * xvals + b

In [45]:
print(f"The self propelled speed should be {xintercept} mm/s")

The self propelled speed should be 395.0973994083102 mm/s


In [44]:
fig = make_subplots(rows = 1, cols = 1,
                   shared_xaxes=True)
fig.add_trace(
    go.Scatter(x = flowspeeds, y = meanyforce, mode="markers", name="y force"),
    row=1, col=1)
fig.add_trace(
    go.Scatter(x = xvals, y = predforce, mode="lines", name="fit y force"),
    row=1, col=1)
